### Import and Load Dataset

In [1]:
import pandas as pd
from darts import TimeSeries

In [2]:
df = pd.read_csv("../Data/cleaned_timeseries.csv")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

series = TimeSeries.from_dataframe(df, time_col="date", value_cols="unit_sales")

In [3]:
# --- Ensure daily frequency and no missing dates (fix interpolation) ---
df = df.sort_values("date").copy()

full_idx = pd.date_range(df["date"].min(), df["date"].max(), freq="D")

df = (
    df.set_index("date")
      .reindex(full_idx)
      .rename_axis("date")
)

# Interpolate while we still have a DatetimeIndex ✅
df["unit_sales"] = df["unit_sales"].interpolate(method="time")

# If there are NaNs at the start/end, fill them (optional but common)
df["unit_sales"] = df["unit_sales"].bfill().ffill()

# Back to regular columns
df = df.reset_index()

# rebuild Darts TimeSeries
series = TimeSeries.from_dataframe(df, time_col="date", value_cols="unit_sales")

In [4]:
print(series.time_index.min(), series.time_index.max())
print(series.time_index[:3])

2013-01-02 00:00:00 2014-03-31 00:00:00
DatetimeIndex(['2013-01-02', '2013-01-03', '2013-01-04'], dtype='datetime64[ns]', name='date', freq='D')


In [5]:
train_series = series.slice(pd.Timestamp("2013-01-02"), pd.Timestamp("2013-12-31"))
test_series  = series.slice(pd.Timestamp("2014-01-01"), pd.Timestamp("2014-03-31"))

In [6]:
print("Train:", train_series.time_index.min(), "→", train_series.time_index.max(), "|", len(train_series))
print("Test :", test_series.time_index.min(),  "→", test_series.time_index.max(),  "|", len(test_series))

Train: 2013-01-02 00:00:00 → 2013-12-31 00:00:00 | 364
Test : 2014-01-01 00:00:00 → 2014-03-31 00:00:00 | 90


### set up metrics + results table

In [7]:
import pandas as pd
from darts.metrics import mae, rmse, smape

results = []

def add_result(model_name, actual, forecast):
    results.append({
        "Model": model_name,
        "MAE": float(mae(actual, forecast)),
        "RMSE": float(rmse(actual, forecast)),
        "sMAPE": float(smape(actual, forecast))
    })

Support for PyTorch based likelihood models not available. To enable them, install "darts[torch]" or "darts[all]" (with pip); or "u8darts-torch" or "u8darts-all" (with conda).


### Naive Seasonal

In [8]:
from darts.models import NaiveSeasonal

naive = NaiveSeasonal(K=7)

naive.fit(train_series)

forecast_naive = naive.predict(len(test_series))

add_result("NaiveSeasonal", test_series, forecast_naive)

print("NaiveSeasonal MAE:", mae(test_series, forecast_naive))
print("NaiveSeasonal RMSE:", rmse(test_series, forecast_naive))
print("NaiveSeasonal sMAPE:", smape(test_series, forecast_naive))

Support for Torch based models not available. To enable them, install "darts[torch]" or "darts[all]" (with pip); or "u8darts-torch" or "u8darts-all" (with conda).
Importing plotly failed. Interactive plots will not work.
The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.


NaiveSeasonal MAE: 180.73333333333332
NaiveSeasonal RMSE: 233.084867710359
NaiveSeasonal sMAPE: 51.950032301069434


### ETS — Exponential Smoothing

In [9]:
from darts.models import ExponentialSmoothing

ets = ExponentialSmoothing()

ets.fit(train_series)

forecast_ets = ets.predict(len(test_series))

add_result("ETS", test_series, forecast_ets)

print("ETS MAE:", mae(test_series, forecast_ets))
print("ETS RMSE:", rmse(test_series, forecast_ets))
print("ETS sMAPE:", smape(test_series, forecast_ets))

ETS MAE: 98.42344842914801
ETS RMSE: 150.4128069771925
ETS sMAPE: 21.95346325612051


### ARIMA

In [10]:
from darts.models import ARIMA

arima = ARIMA(p=3, d=0, q=3)

arima.fit(train_series)

forecast_arima = arima.predict(len(test_series))

add_result("ARIMA", test_series, forecast_arima)

print("ARIMA MAE:", mae(test_series, forecast_arima))
print("ARIMA RMSE:", rmse(test_series, forecast_arima))
print("ARIMA sMAPE:", smape(test_series, forecast_arima))

ARIMA MAE: 116.41399321987812
ARIMA RMSE: 162.19684040447078
ARIMA sMAPE: 25.9852665027976


D:\data analytics\TimeSeries\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


### SARIMA

In [11]:
from darts.models import ARIMA

sarima = ARIMA(
    p=3,
    d=0,
    q=3,
    seasonal_order=(1,1,1,7)  # weekly seasonality
)

sarima.fit(train_series)

forecast_sarima = sarima.predict(len(test_series))

add_result("SARIMA", test_series, forecast_sarima)

print("SARIMA MAE:", mae(test_series, forecast_sarima))
print("SARIMA RMSE:", rmse(test_series, forecast_sarima))
print("SARIMA sMAPE:", smape(test_series, forecast_sarima))

D:\data analytics\TimeSeries\.venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
D:\data analytics\TimeSeries\.venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


SARIMA MAE: 98.96450125037929
SARIMA RMSE: 149.997704171094
SARIMA sMAPE: 22.086330619778845


D:\data analytics\TimeSeries\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [15]:
# Convert Darts TimeSeries -> pandas Series
y_true = pd.Series(
    test_series.values().reshape(-1),
    index=test_series.time_index
)

y_pred = pd.Series(
    forecast_sarima.values().reshape(-1),
    index=forecast_sarima.time_index
)

In [25]:
from pathlib import Path

OUT_DIR = Path("../models")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_sarima = pd.DataFrame({
    "actual": y_true,
    "sarima": y_pred
}).dropna()

file_path = OUT_DIR / "sarima_forecast.csv"

df_sarima.to_csv(file_path)

print("Saved to:", file_path.resolve())

Saved to: D:\data analytics\TimeSeries\models\sarima_forecast.csv


### Theta

In [18]:
print("Min train value:", train_series.values().min())
print("Min test value :", test_series.values().min())

Min train value: 0.0
Min test value : 0.0


In [19]:
## Theta requires strictly positive values; since the series contains zeros, we shift the series by +1 during training and shift predictions back.
from darts.models import Theta

shift = 1  # makes zeros -> 1 (strictly positive)

train_pos = train_series + shift
test_pos  = test_series + shift

theta = Theta()
theta.fit(train_pos)

forecast_theta_pos = theta.predict(len(test_pos))

# shift back to original scale
forecast_theta = forecast_theta_pos - shift

add_result("Theta", test_series, forecast_theta)

print("Theta MAE:", mae(test_series, forecast_theta))
print("Theta RMSE:", rmse(test_series, forecast_theta))
print("Theta sMAPE:", smape(test_series, forecast_theta))

Theta MAE: 98.72088321362241
Theta RMSE: 150.03847343353254
Theta sMAPE: 22.021881065365186


### Prophet (Additive)

In [20]:
from darts.models import Prophet

prophet_add = Prophet(seasonality_mode="additive")

prophet_add.fit(train_series)

forecast_prophet = prophet_add.predict(len(test_series))

add_result("Prophet Additive", test_series, forecast_prophet)

print("Prophet MAE:", mae(test_series, forecast_prophet))
print("Prophet RMSE:", rmse(test_series, forecast_prophet))
print("Prophet sMAPE:", smape(test_series, forecast_prophet))

23:13:01 - cmdstanpy - INFO - Chain [1] start processing
23:13:01 - cmdstanpy - INFO - Chain [1] done processing


Prophet MAE: 98.35812955750713
Prophet RMSE: 150.46909176906706
Prophet sMAPE: 21.93493835075493


#### Statistical Models Comparison

In [21]:
# Create dataframe
results_df = pd.DataFrame(results)

# Remove duplicates (in case a model ran twice)
results_df = results_df.drop_duplicates(subset="Model")

# Sort by performance
results_df = results_df.sort_values("MAE").reset_index(drop=True)

# Show table
results_df

,Model,MAE,RMSE,sMAPE
0,Prophet Additive,98.358130,150.469092,21.934938
1,ETS,98.423448,150.412807,21.953463
2,Theta,98.720883,150.038473,22.021881
3,SARIMA,98.964501,149.997704,22.086331
4,ARIMA,116.413993,162.196840,25.985267
5,NaiveSeasonal,180.733333,233.084868,51.950032


In [22]:
results_df.to_csv("../models/metrics_statistical.csv", index=False)

### Classical statistical models show very similar performance, with Prophet Additive achieving the lowest MAE. The small differences between ETS, Theta, and SARIMA indicate that the sales series exhibits stable seasonal patterns that can be captured effectively by multiple statistical approaches.